# Agentes - ReAct

## 1. Fundamentos de ReAct

## Fundamentos de ReAct

**ReAct** (Reasoning and Acting) es un patrón de diseño para agentes de IA que combina la **razón** (pensamiento) con la **acción** (uso de herramientas) para resolver tareas complejas. Su objetivo principal es permitir que los modelos de lenguaje (LLMs) no solo generen respuestas, sino que también interactúen con el entorno y utilicen recursos externos de manera efectiva.

### Componentes Clave:

1.  **Razonamiento (Reasoning):**
    *   El agente genera pensamientos internos para planificar, reflexionar y descomponer problemas. Esto implica:
        *   **Planificación:** Decidir qué pasos tomar para alcanzar un objetivo.
        *   **Análisis:** Evaluar la información disponible y los resultados de las acciones.
        *   **Reflexión:** Aprender de errores o resultados inesperados para ajustar estrategias.

2.  **Actuación (Acting):**
    *   El agente ejecuta acciones utilizando **herramientas** (tools) específicas. Estas herramientas pueden ser funciones de código, APIs, bases de datos, o cualquier recurso externo que el agente pueda invocar. Cada herramienta tiene una descripción clara de su propósito y cómo usarla.

### Ciclo de Operación de ReAct:

El agente ReAct opera en un ciclo iterativo:

1.  **Observación (Observation):** El agente recibe una consulta del usuario o el resultado de una acción previa.
2.  **Pensamiento (Thought):** Basado en la observación, el agente "piensa" o razona sobre:
    *   Cuál es el objetivo actual.
    *   Qué información necesita.
    *   Qué herramientas son relevantes.
    *   Cuál sería el siguiente paso lógico (planificación).
3.  **Acción (Action):** El agente selecciona una herramienta apropiada y genera los argumentos necesarios para invocarla.
4.  **Resultado (Observation):** La herramienta se ejecuta y devuelve un resultado, que se convierte en la siguiente "observación" para el agente.

Este ciclo se repite hasta que el agente considera que la tarea ha sido completada o que no puede avanzar más.

### Beneficios de ReAct:

*   **Mayor Capacidad de Resolución de Problemas:** Permite a los LLMs abordar tareas que requieren más que solo generación de texto, como cálculos, búsquedas de información en tiempo real o interacción con sistemas externos.
*   **Transparencia:** El proceso de pensamiento del agente puede ser inspeccionado, lo que facilita entender por qué tomó ciertas decisiones.
*   **Flexibilidad:** Puede integrar una amplia variedad de herramientas para extender las capacidades del agente.
*   **Robustez:** Permite al agente recuperarse de errores o información incompleta al razonar sobre los resultados y ajustar su plan.

## 2. Instalación de Paquetes

In [ ]:
!pip install -U langchain langchain-openrouter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.6 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18


In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


## 3. Configuración del Entorno y Modelo

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from google.colab import userdata
import os
import requests
from openai import OpenAI

In [ ]:
os.environ["OPENROUTER_KEY"] = userdata.get('OPENROUTER_KEY')

In [ ]:
# Configurar el cliente para OpenRouter
# Inicializar el cliente leyendo desde la variable de entorno
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ.get("OPENROUTER_KEY")
)

In [ ]:
model = ChatOpenAI(
    model="cohere/north-mini-code:free",
    api_key=os.environ.get("OPENROUTER_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

## 4. Definición de Herramientas (Tools)

In [ ]:
@tool
def calcular(expresion: str) -> str:
    """Calcula una expresión matemática."""
    return str(eval(expresion))


In [ ]:
@tool
def buscar_poblacion(pais: str) -> str:
    """Busca la población aproximada de un país."""

    poblaciones = {
        "argentina": 46_000_000,
        "brasil": 212_000_000,
        "chile": 20_000_000,
        "uruguay": 3_500_000
    }

    return str(poblaciones.get(
        pais.lower(),
        "No encontré información"
    ))

## 5. Creación del Agente ReAct

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[
        calcular,
        buscar_poblacion
    ],
    system_prompt="""
    Sos un asistente que puede utilizar herramientas.

    Utilizá buscar_poblacion cuando necesites información
    sobre población.

    Utilizá calcular cuando necesites realizar operaciones
    matemáticas.

    Podés utilizar varias herramientas si la pregunta lo requiere.
    """
)

## 6. Invocación del Agente y Análisis de Resultados

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            ¿Cuál es la población de Brasil y cuántas veces
            supera aproximadamente a la población de Argentina?
            """
        }
    ]
})

In [ ]:
for message in result["messages"]:
    print("\n---")
    print(message)


---
content='\n            ¿Cuál es la población de Brasil y cuántas veces\n            supera aproximadamente a la población de Argentina?\n            ' additional_kwargs={} response_metadata={} id='3a003e74-64f7-41ba-9d8c-7da81d1cefb3'

---
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 134, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 113, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'cohere/north-mini-code:free', 'system_fingerprint': None, 'id': 'gen-1789661630-1xrfUsUyVEPm2ivNiTaq', 'finish_reason': 'tool_ca

In [ ]:
for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)
    print(getattr(message, "tool_calls", None))
    print("---")

HumanMessage

            ¿Cuál es la población de Brasil y cuántas veces
            supera aproximadamente a la población de Argentina?
            
None
---
AIMessage

[{'name': 'buscar_poblacion', 'args': {'pais': 'Brasil'}, 'id': 'buscar_poblacion_s50s1vt9r2qk', 'type': 'tool_call'}, {'name': 'buscar_poblacion', 'args': {'pais': 'Argentina'}, 'id': 'buscar_poblacion_et3k71xxh06h', 'type': 'tool_call'}]
---
ToolMessage
212000000
None
---
ToolMessage
46000000
None
---
AIMessage

[{'name': 'calcular', 'args': {'expresion': '212000000 / 46000000'}, 'id': 'calcular_9cj0dp21pm3y', 'type': 'tool_call'}]
---
ToolMessage
4.608695652173913
None
---
AIMessage
Basándome en la información de población que encontré:

- **Población de Brasil:** 212,000,000 personas
- **Población de Argentina:** 46,000,000 personas

La población de Brasil supera aproximadamente **4.6 veces** a la población de Argentina (212,000,000 ÷ 46,000,000 ≈ 4.61).

Esto significa que Brasil tiene alrededor de 4.6 veces más 